# 24 — Zero-Shot Classification
**Goal:** Classify text into categories without any training data.

Zero-shot classification labels text with categories the model has *never been trained on*. Instead of learning a classifier, it reuses **Natural Language Inference (NLI)**: the model decides whether a candidate label's claim ("This text is about Data Science") is entailed by the input. No labeled data, no fine-tuning — just a list of candidate labels per query.

**Why it matters for resumes / ATS:** resume data arrives unlabeled — sections, skill types, and JD-requirement severity all need categories, and hand-labeling costs time and drifts between job markets. Zero-shot turns every classification problem into a label-engineering problem: write good candidate labels and the model sorts the text. That is the difference between a hard-coded section regex and a classifier that understands "Published 5 papers…" is Publications.

## 1. What is Zero-Shot Classification?

NLI asks: given a **premise** (the text) and a **hypothesis** (a label phrased as a claim), does the premise entail the hypothesis? Zero-shot classification scores every label's hypothesis against the text and returns the scores — the best label wins. The trick is that entailment reasoning transfers: the model doesn't "recognize" TensorFlow, it reasons that "Built ML models with TensorFlow" implies "This is about machine learning".

**What the code does:** prints a cheat-sheet showing the premise/hypothesis mechanics and lists the resume use cases this chapter exercises: section classification (Experience/Education/Skills), skill categorization (technical/soft/tool/domain), requirement severity (must-have/nice-to-have), and seniority detection. The 0.92 score in the example is illustrative — real scores depend on model and text.

**Why it matters for resumes / ATS:** one model, no training loop, and the label list is the entire "config" — swap `["must-have", "nice-to-have", "preferred"]` for `["Junior", "Mid", "Senior", "Lead"]` and the same pipeline classifies something completely different.

In [ ]:
print('''Zero-shot classification uses NLI (Natural Language Inference) to classify text:

  Text: "Built ML models with TensorFlow"
  Hypothesis: "This is about machine learning"
  → Entailment (score: 0.92)

  No training data needed! Just provide candidate labels.
  
Uses in resume analysis:
  - Classify resume sections (Experience, Education, Skills)
  - Categorize skills (technical, soft, domain)
  - Classify job requirements (must-have, nice-to-have)
  - Detect seniority levels (Junior, Mid, Senior, Lead)''')

## 2. Setting Up the Pipeline

`transformers.pipeline("zero-shot-classification")` bundles model + tokenizer + scoring into one callable. The default here is `facebook/bart-large-mnli` — a BART model fine-tuned on the MultiNLI entailment corpus — the standard zero-shot workhorse (~1.6 GB on first download, CPU-friendly at inference).

**What the code does:** loads the pipeline, then classifies one resume line ("Built NLP pipelines processing 10M documents daily using Python and TensorFlow") against four labels. `result['labels']` and `result['scores']` come back **sorted best-first**, and the scores are normalized to sum to 1. Expected: **Data Science** first — the text is dense with ML/NLP vocabulary — with Software Engineering second, and Management/Research trailing; the print loop shows all four with their scores.

**Try it:** add `"Data Engineering"` as a fifth label and watch the score mass redistribute — labels compete, so adding a strong candidate lowers the others.

In [ ]:
from transformers import pipeline
print("Loading zero-shot classifier...")
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
print("Model loaded.")

# Test on a resume line
text = "Built NLP pipelines processing 10M documents daily using Python and TensorFlow"
candidate_labels = ["Software Engineering", "Data Science", "Management", "Research"]
result = classifier(text, candidate_labels)
print(f"Text: {text[:50]}...")
print("\nClassification:")
for label, score in zip(result['labels'], result['scores']):
    print(f"  {label:20s} {score:.3f}")

## 3. Resume Section Classification

Section detection is usually regex-driven ("EDUCATION" headers) and breaks the moment formatting varies. Zero-shot sectioning replaces the regex with semantics: any line can be scored against the section vocabulary.

**What the code does:** four resume lines, each tagged with its true section, are classified against five categories (`Experience`, `Education`, `Skills`, `Publications`, `Summary`). The cell prints predicted vs true with a ✓ when they agree. Expected: all four lines classified correctly — "Senior Data Scientist at Google, 2020-Present" → **Experience**, "M.S. in Computer Science, Stanford University" → **Education**, the bare skill list → **Skills**, and "Published 5 papers in top-tier NLP conferences" → **Publications**. The cues the model leans on (dates/companies vs degrees vs tool lists) are exactly the ones a human reader uses.

**Try it:** reorder `categories` and note the prediction is stable — zero-shot output depends on label *wording*, not label order.

In [ ]:
resume_lines = [
    ("Senior Data Scientist at Google, 2020-Present", "Experience"),
    ("M.S. in Computer Science, Stanford University", "Education"),
    ("Python, TensorFlow, PyTorch, SQL, AWS", "Skills"),
    ("Published 5 papers in top-tier NLP conferences", "Publications"),
]
categories = ["Experience", "Education", "Skills", "Publications", "Summary"]

print("Section classification:")
for text, true_label in resume_lines:
    result = classifier(text, categories)
    predicted = result['labels'][0]
    correct = "✓" if predicted == true_label else " "
    print(f"  {correct} '{text[:45]:45s}' → {predicted:12s} (true: {true_label})")

## 4. Skill Category Detection

Skill taxonomies (technical / soft / tool / domain) are a classic hand-curated list that rots over time. Zero-shot categorizes individual skills against the taxonomy with no list maintenance.

**What the code does:** eight skills are each classified against `["technical", "soft skill", "tool", "domain knowledge"]`; the top label and its confidence are printed. Expected pattern:
- **technical** — Python, TensorFlow, PyTorch;
- **soft skill** — Team Leadership, Communication, Project Management, Critical Thinking;
- **tool** — Docker (and Kubernetes-adjacent infrastructure);
- **domain knowledge** — little or nothing in this list.

The honest caveat: single-word inputs give the NLI model almost no context, so expect **lower confidence** for short skills like "Python" than for the sentence-length examples of §2–3. Zero-shot is strongest on text that carries its own context.

In [ ]:
skills = ["Python", "Team Leadership", "TensorFlow", "Communication",
              "Docker", "Project Management", "PyTorch", "Critical Thinking"]
skill_categories = ["technical", "soft skill", "tool", "domain knowledge"]

print("Skill categorization:")
for skill in skills:
    result = classifier(skill, skill_categories)
    cat = result['labels'][0]
    conf = result['scores'][0]
    print(f"  {skill:20s} → {cat:15s} ({conf:.2f})")

## 5. Requirement Classification

JDs mix hard requirements ("5+ years Python"), soft asks ("strong communication"), and hedged preferences ("PhD preferred"). Sorting these automatically changes what a matcher can promise — must-haves gate, nice-to-haves rank.

**What the code does:** five JD requirements are classified against `["must-have", "nice-to-have", "preferred"]`. Expected:
- "5+ years experience in Python" → **must-have**;
- "Experience with AWS or GCP" → **must-have**;
- "Strong communication skills" → **nice-to-have** (or must-have — the boundary is fuzzy);
- "Ability to work in fast-paced environment" → **nice-to-have**;
- "PhD in Computer Science preferred" → **preferred** — the word "preferred" in the input is the strongest cue the model can get.

The fuzzier cases are the interesting ones: severity is genuinely ambiguous, and the confidence score tells you when to route to a human instead of auto-deciding.

In [ ]:
jd_requirements = [
    "5+ years experience in Python",
    "Strong communication skills",
    "PhD in Computer Science preferred",
    "Experience with AWS or GCP",
    "Ability to work in fast-paced environment",
]
req_types = ["must-have", "nice-to-have", "preferred"]

print("Requirement classification:")
for req in jd_requirements:
    result = classifier(req, req_types)
    print(f"  '{req[:45]:45s}' → {result['labels'][0]:12s} ({result['scores'][0]:.2f})")

## Key Insight: Zero-shot classification works surprisingly well for resume tasks. Requires no training data.

**NLI turns classification into label engineering — the label list is the only thing you tune.**

One `pipeline` call labels sections, skills, and JD requirements without a single labeled example, and the same model is reused across all three tasks by swapping candidate labels. The limits are real: short inputs get lower confidence, and label wording matters more than label order. Ch. 25 keeps this in perspective — a classifier is only as production-ready as the error handling around it, which is exactly the next chapter's subject.